# 07 - Entanglement swapping

Connect Alice and Charlie through two Bell pairs and Bob's joint measurement. Inspect every outcome, the missing-record ensemble, and classical feed-forward. The model assumes two perfect available links and ideal local operations; it does not simulate a physical network delivery rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from quantum_ed.density import fidelity_pure, partial_trace_two_qubits, rho_from_ket
from quantum_ed.gates import X
from quantum_ed.networks import OUTCOMES, conditional_endpoint_state, endpoint_density
from quantum_ed.states import bell_phi_plus

## Four measurement outcomes

Tensor order is A, Bob-left, Bob-right, C. Outcome zx lists Bob-left first: z controls the phase correction and x the bit correction. Charlie applies X for x=1, then Z for z=1. No outcome is discarded.

In [ ]:
target_state = bell_phi_plus()
target_density = rho_from_ket(target_state)

print("Bob   Probability   Fidelity before   Fidelity after")
for outcome in OUTCOMES:
    probability, raw_state = conditional_endpoint_state(outcome, corrected=False)
    _, corrected_state = conditional_endpoint_state(outcome)
    before = fidelity_pure(target_state, rho_from_ket(raw_state))
    after = fidelity_pure(target_state, rho_from_ket(corrected_state))
    print(f"{outcome}    {probability:.3f}         {before:.3f}             {after:.3f}")
    assert np.isclose(probability, 0.25)
    assert np.allclose(rho_from_ket(corrected_state), target_density)

## Why endpoint Z-basis probabilities are insufficient

For Bob's outcome 10, the raw state is Phi-minus. It has the same computational-basis probabilities as Phi-plus, but the sign of its relative phase changes the XX correlation.

In [ ]:
_, phase_flipped_state = conditional_endpoint_state("10", corrected=False)
print("Phi-plus probabilities:", np.abs(target_state.ravel()) ** 2)
print("Phi-minus probabilities:", np.abs(phase_flipped_state.ravel()) ** 2)
xx = np.kron(X, X)
print("Phi-plus XX:", float(np.trace(target_density @ xx).real))
print("Phi-minus XX:", float(np.trace(rho_from_ket(phase_flipped_state) @ xx).real))
assert np.allclose(np.abs(target_state) ** 2, np.abs(phase_flipped_state) ** 2)

## Discarding the record versus using it

Without Bob's record and before correction, averaging over outcomes gives I4/4. After outcome-dependent corrections, all branches give the Phi-plus projector. The figure shows real matrix entries; both matrices in this example have zero imaginary part.

In [ ]:
unrecorded_density = endpoint_density()
corrected_density = endpoint_density(corrected=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
labels = ["00", "01", "10", "11"]
for axis, density, title in zip(
    axes, [unrecorded_density, corrected_density],
    ["Outcome record unavailable", "After classical feed-forward"]
):
    image = axis.imshow(density.real, vmin=0, vmax=0.5, cmap="Blues")
    axis.set_xticks(range(4), labels)
    axis.set_yticks(range(4), labels)
    axis.set_xlabel("Column basis state")
    axis.set_ylabel("Row basis state")
    axis.set_title(title)
    for row in range(4):
        for column in range(4):
            axis.text(column, row, f"{density[row, column].real:.2f}",
                      ha="center", va="center",
                      color="white" if density[row, column].real > 0.3 else "black")
fig.colorbar(image, ax=axes, label="Real density-matrix entry")
plt.show()

## Local states and communication

Both endpoints individually have density matrix I2/2, before and after feed-forward. The corrected resource changes their joint correlations. It requires the classical outcome; the ideal protocol does not transmit a usable message instantaneously.

In [ ]:
assert np.allclose(unrecorded_density, np.eye(4) / 4)
assert np.allclose(corrected_density, target_density)
for density in [unrecorded_density, corrected_density]:
    for endpoint in [0, 1]:
        local_density = partial_trace_two_qubits(density, keep=endpoint)
        assert np.allclose(local_density, np.eye(2) / 2)
print("All swapping checks passed.")

## Practice and boundaries

1. Keep only outcome 00: its selection probability is 1/4. Explain why reporting only its conditional fidelity hides discarded trials.
2. Omit only the Z correction and identify which branches acquire an incorrect relative phase.
3. List the additional parameters needed to model photon loss and memory waiting.

The SDK experiment uses deferred measurement to represent feed-forward inside one simulator. Real nodes require messaging and timing assumptions. No repeater scheduling or QKD security analysis is implemented. For experimental context, see [Hermans et al., Nature (2022)](https://arxiv.org/abs/2110.11373).